# 01 · Silver – reference tables
**Phase 4 – Silver layer.** Cleans the small reference tables `cities` and `cuisine_category_translation` from Bronze and saves them to `glovo_dev.silver`.

**Rules for Silver**
- Correct types, consistent values, no duplicates.
- Bad rows are not deleted: they go to `glovo_dev.silver.quarantine` with a reason.
- Audit columns from Bronze are kept, so every row can still be traced to its source file.

## 1. Settings

In [0]:
from pyspark.sql import functions as F
CATALOG = "glovo_dev"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

## 2. Look at the Bronze data

In [0]:
cities_bronze = spark.table(f"{BRONZE}.cities")
display(cities_bronze)

city_id,city_name,voivodeship,population,latitude,longitude,city_tier,_rescued_data,_ingested_at,_source_file
CTY001,Warszawa,mazowieckie,1863056,52.2297,21.0122,tier_1,null,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY002,Kraków,malopolskie,804237,50.0647,19.945,tier_1,null,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY003,Łódź,lodzkie,664071,51.7592,19.456,tier_1,null,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY004,Wrocław,dolnoslaskie,674132,51.1079,17.0385,tier_1,null,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY005,Poznań,wielkopolskie,546859,52.4064,16.9252,tier_1,null,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY006,Gdańsk,pomorskie,470907,54.352,18.6466,tier_2,null,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY007,Szczecin,zachodniopomorskie,396168,53.4285,14.5528,tier_2,null,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY008,Bydgoszcz,kujawsko-pomorskie,348190,53.1235,18.0084,tier_2,null,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY009,Lublin,lubelskie,336339,51.2465,22.5684,tier_2,null,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY010,Białystok,podlaskie,297554,53.1325,23.1688,tier_2,null,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet


## 3. City match key
Imported from the shared module `src/glovo/silver/cleaning.py` (unit-tested in CI), so every Silver notebook uses the same logic.
`" ŁÓDŹ "`, `"Lodz"` and `"Łódź"` all become `"lodz"`.

In [0]:
import os
import sys

sys.path.append(os.path.abspath("../../src"))
from glovo.silver.cleaning import city_key

# quick test
test = spark.createDataFrame([(" ŁÓDŹ ",), ("Lodz",), ("łódź",), ("KRAKÓW",)], ["city"])
display(test.withColumn("key", city_key(F.col("city"))))

city,key
ŁÓDŹ,lodz
Lodz,lodz
łódź,lodz
KRAKÓW,krakow


## 4. Silver cities
Keeps the official city list, adds `city_key` for matching, and checks that every key is unique.

In [0]:
cities_silver = (
    cities_bronze
    .withColumn("city_name", F.trim(F.col("city_name")))
    .withColumn("city_key", city_key(F.col("city_name")))
    .select(
        "city_id", "city_name", "city_key", "voivodeship", "population",
        "latitude", "longitude", "city_tier",
        "_ingested_at", "_source_file",
    )
)

display(cities_silver)

city_id,city_name,city_key,voivodeship,population,latitude,longitude,city_tier,_ingested_at,_source_file
CTY001,Warszawa,warszawa,mazowieckie,1863056,52.2297,21.0122,tier_1,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY002,Kraków,krakow,malopolskie,804237,50.0647,19.945,tier_1,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY003,Łódź,lodz,lodzkie,664071,51.7592,19.456,tier_1,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY004,Wrocław,wroclaw,dolnoslaskie,674132,51.1079,17.0385,tier_1,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY005,Poznań,poznan,wielkopolskie,546859,52.4064,16.9252,tier_1,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY006,Gdańsk,gdansk,pomorskie,470907,54.352,18.6466,tier_2,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY007,Szczecin,szczecin,zachodniopomorskie,396168,53.4285,14.5528,tier_2,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY008,Bydgoszcz,bydgoszcz,kujawsko-pomorskie,348190,53.1235,18.0084,tier_2,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY009,Lublin,lublin,lubelskie,336339,51.2465,22.5684,tier_2,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet
CTY010,Białystok,bialystok,podlaskie,297554,53.1325,23.1688,tier_2,2026-11-25T19:27:44.443Z,abfss://source@stglovolake122.dfs.core.windows.net/cities/glovo_cities_dataset.parquet


In [0]:
total = cities_silver.count()
unique_keys = cities_silver.select("city_key").distinct().count()
print(f"rows: {total}, unique keys: {unique_keys}")

assert total == unique_keys, "Two cities have the same key!"

rows: 36, unique keys: 36


## Saving the Silver Cities Table

In [0]:
(
    cities_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.cities")

)

In [0]:
%sql
SELECT city_name, city_key, city_tier
FROM glovo_dev.silver.cities
ORDER BY population DESC
LIMIT 10

city_name,city_key,city_tier
Warszawa,warszawa,tier_1
Kraków,krakow,tier_1
Wrocław,wroclaw,tier_1
Łódź,lodz,tier_1
Poznań,poznan,tier_1
Gdańsk,gdansk,tier_2
Szczecin,szczecin,tier_2
Bydgoszcz,bydgoszcz,tier_2
Lublin,lublin,tier_2
Białystok,bialystok,tier_2


## 5. Silver cuisine categories
Cleans the cuisine codes, adds a readable name for reports, and checks that each code is unique.

In [0]:
cuisine_bronze = spark.table(f"{BRONZE}.cuisine_category_translation")
display(cuisine_bronze)

cuisine_category_pl,cuisine_category_en,_rescued_data,_ingested_at,_source_file
pizza,pizza,null,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
burger,burgers,null,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
kebab,kebab,null,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
sushi,sushi,null,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
kuchnia_polska,polish_cuisine,null,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
kuchnia_chinska,chinese_cuisine,null,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
kuchnia_wloska,italian_cuisine,null,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
kuchnia_indyjska,indian_cuisine,null,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
kuchnia_wegetarianska,vegetarian_cuisine,null,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
zdrowa_zywnosc,healthy_food,null,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet


## Data Quality Check: Validate Uniqueness and Completeness of Cuisine Category Translations

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT cuisine_category_pl) AS unique_pl,
    COUNT(DISTINCT cuisine_category_en) AS unique_en,

    COUNT_IF(cuisine_category_pl IS NULL) AS null_pl,
    COUNT_IF(cuisine_category_en IS NULL) AS null_en,

    CASE
        WHEN COUNT(*) = COUNT(DISTINCT cuisine_category_pl)
         AND COUNT(*) = COUNT(DISTINCT cuisine_category_en)
         AND COUNT_IF(cuisine_category_pl IS NULL) = 0
         AND COUNT_IF(cuisine_category_en IS NULL) = 0
        THEN 'PASS'
        ELSE 'FAIL: Duplicate or NULL cuisine categories!'
    END AS check_result

FROM glovo_dev.bronze.cuisine_category_translation;

total_rows,unique_pl,unique_en,null_pl,null_en,check_result
20,20,20,0,0,PASS


In [0]:
cuisine_silver = (
    cuisine_bronze
    .withColumn("cuisine_category_pl", F.lower(F.trim(F.col("cuisine_category_pl"))))
    .withColumn("cuisine_category_en", F.lower(F.trim(F.col("cuisine_category_en"))))
    .withColumn("cuisine_name", F.initcap(F.regexp_replace(F.col("cuisine_category_en"), "_", " ")))
    .select("cuisine_category_pl", "cuisine_category_en", "cuisine_name", "_ingested_at", "_source_file")
)

display(cuisine_silver)

total = cuisine_silver.count()
unique_codes = cuisine_silver.select("cuisine_category_pl").distinct().count()
print(f"rows: {total}, unique codes: {unique_codes}")
assert total == unique_codes, "Duplicate cuisine codes!"

cuisine_category_pl,cuisine_category_en,cuisine_name,_ingested_at,_source_file
pizza,pizza,Pizza,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
burger,burgers,Burgers,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
kebab,kebab,Kebab,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
sushi,sushi,Sushi,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
kuchnia_polska,polish_cuisine,Polish Cuisine,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
kuchnia_chinska,chinese_cuisine,Chinese Cuisine,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
kuchnia_wloska,italian_cuisine,Italian Cuisine,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
kuchnia_indyjska,indian_cuisine,Indian Cuisine,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
kuchnia_wegetarianska,vegetarian_cuisine,Vegetarian Cuisine,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet
zdrowa_zywnosc,healthy_food,Healthy Food,2026-11-25T19:27:58.027Z,abfss://source@stglovolake122.dfs.core.windows.net/cuisine_category_translation/glovo_cuisine_category_translation.parquet


rows: 20, unique codes: 20


In [0]:
(
    cuisine_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.cuisine_categories")

)

## 6. Result

In [0]:
%sql
SELECT 'cities' AS table_name, COUNT(*) AS row_count FROM glovo_dev.silver.cities
UNION ALL
SELECT 'cuisine_categories', COUNT(*) FROM glovo_dev.silver.cuisine_categories

table_name,row_count
cities,36
cuisine_categories,20
